# Vendor Performance Analytical Project

In [ ]:
import pandas as pd
import os
import numpy as np
from sqlalchemy import create_engine
import logging
import time

#  Logging setup
logging.basicConfig(
    filename="logs/ingestion_db.log",
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)

#  MySQL connection
engine = create_engine('mysql+pymysql://root:<password>@localhost:3306/inventorydb')

#  Ingest function
def ingest_db(df, table_name, engine):
    """This function will ingest dataframe into database table"""
    df.to_sql(name=table_name, con=engine, if_exists='replace', index=False)

#  Main load function
def load_raw_data():
    """Load CSV files and ingest into database"""
    
    start = time.time()
    
    files = os.listdir('data')
    print(" Files found:", files)
    
    for file in files:
        
        #  Skip hidden/system files
        if file.startswith('.'):
            continue
        
        #  Only CSV files
        if file.lower().endswith('.csv'):
            try:
                print(f"\n Processing: {file}")
                
                #  Read CSV
                df = pd.read_csv(os.path.join('data', file), encoding='latin1')
                
                #  Clean column names
                df.columns = df.columns.str.strip()
                
                #  IMPORTANT: Handle inf values
                df.replace([np.inf, -np.inf], np.nan, inplace=True)
                
                #  Optional: Fill NaN (if needed)
                # df.fillna(0, inplace=True)
                
                # Table name from file
                table_name = file[:-4]
                
                #  Ingest into DB
                ingest_db(df, table_name, engine)
                
                logging.info(f"Ingesting {file} in db")
                print(f" Success: {file}")
            
            except Exception as e:
                print(f" Error in {file}: {e}")
                logging.error(f"Error in {file}: {e}")
    
    end = time.time()
    total_time = (end - start) / 60
    
    logging.info('----------------Ingestion Complete----------------')
    logging.info(f'Total Time Taken: {total_time} minutes')
    
    print(f"\n Total Time Taken: {total_time:.2f} minutes")


# 🔹 Run script
if __name__ == '__main__':
    load_raw_data()

📂 Files found: ['.ipynb_checkpoints', 'begin_inventory.csv', 'end_inventory.csv', 'purchases.csv', 'purchase_prices.csv', 'sales.csv', 'vendor_invoice.csv', 'vendor_sales_summary.csv']

🔄 Processing: begin_inventory.csv
✅ Success: begin_inventory.csv

🔄 Processing: end_inventory.csv
✅ Success: end_inventory.csv

🔄 Processing: purchases.csv
